# Private dragon generation
Original licensed weights, no secrets in notebook. Technical candidate; artistic acceptance requires separate shape and material review.

In [ ]:
from pathlib import Path
Path('/kaggle/temp').mkdir(parents=True,exist_ok=True)
Path('/kaggle/temp/t4-adaptation.py').write_text('"""Reviewed T4 adaptations; preserve trained BF16/FP16 weights and attention semantics."""\nimport gc,pathlib\n\ndef patch_source(repo):\n    config=repo/\'trellis2/modules/sparse/config.py\'\n    text=config.read_text()\n    old="env_sparse_attn_backend in [\'xformers\', \'flash_attn\', \'flash_attn_3\']"\n    if text.count(old)!=1:raise RuntimeError(\'Sparse config anchor changed\')\n    config.write_text(text.replace(old,old[:-1]+", \'sdpa\']"))\n    target=repo/\'trellis2/modules/sparse/attention/full_attn.py\'\n    text=target.read_text()\n    anchor=\'    else:\\n        raise ValueError(f"Unknown attention module: {config.ATTN}")\'\n    if text.count(anchor)!=1:raise RuntimeError(\'Sparse attention anchor changed\')\n    branch=\'\'\'    elif config.ATTN == \'sdpa\':\n        from torch.nn.functional import scaled_dot_product_attention as sdpa\n        if num_all_args == 1:\n            q, k, v = qkv.unbind(dim=1)\n        elif num_all_args == 2:\n            k, v = kv.unbind(dim=1)\n        outputs = []\n        q_start = kv_start = 0\n        for q_len, kv_len in zip(q_seqlen, kv_seqlen):\n            qi = q[q_start:q_start+q_len].unsqueeze(0).transpose(1, 2)\n            ki = k[kv_start:kv_start+kv_len].unsqueeze(0).transpose(1, 2)\n            vi = v[kv_start:kv_start+kv_len].unsqueeze(0).transpose(1, 2)\n            outputs.append(sdpa(qi, ki, vi).transpose(1, 2).squeeze(0))\n            q_start += q_len\n            kv_start += kv_len\n        out = torch.cat(outputs, dim=0)\n\'\'\'\n    target.write_text(text.replace(anchor,branch+anchor))\n    pipeline=repo/\'trellis2/pipelines/trellis2_image_to_3d.py\'\n    text=pipeline.read_text()\n    old="pipeline.rembg_model = getattr(rembg, args[\'rembg_model\'][\'name\'])(**args[\'rembg_model\'][\'args\'])"\n    if text.count(old)!=1:raise RuntimeError(\'Background loader anchor changed\')\n    pipeline.write_text(text.replace(old,\'pipeline.rembg_model = None  # input was already background-removed\'))\n\ndef install_runtime(model_pins,report,save):\n    import torch,huggingface_hub\n    import torch.nn.functional as F\n    from torch.nn.attention import sdpa_kernel,SDPBackend\n    original_sdpa=F.scaled_dot_product_attention\n    def efficient(q,k,v,*args,**kwargs):\n        if q.is_cuda and q.dtype==torch.bfloat16:\n            with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):\n                output=original_sdpa(q.float(),k.float(),v.float(),*args,**kwargs)\n            return output.to(q.dtype)\n        return original_sdpa(q,k,v,*args,**kwargs)\n    F.scaled_dot_product_attention=efficient\n    original_download=huggingface_hub.hf_hub_download\n    def pinned_download(repo_id,filename,*args,**kwargs):\n        if repo_id not in model_pins:raise ValueError(\'Model outside pinned repositories\')\n        kwargs[\'revision\']=model_pins[repo_id][\'revision\']\n        return original_download(repo_id,filename,*args,**kwargs)\n    huggingface_hub.hf_hub_download=pinned_download\n    from trellis2 import models\n    original_load=models.from_pretrained\n    class LazyModel(torch.nn.Module):\n        def __init__(self,path,kwargs):\n            super().__init__()\n            object.__setattr__(self,\'path\',path);object.__setattr__(self,\'kwargs\',kwargs)\n            object.__setattr__(self,\'loaded\',None);object.__setattr__(self,\'overrides\',{})\n        def _load(self):\n            if self.loaded is None:\n                report[\'loading_model\']=self.path;save()\n                # Constructors initialize CPU tensors before loading the checkpoint.\n                # Preserve the sampling RNG when moving eager construction into a later stage.\n                with torch.random.fork_rng(devices=[]):\n                    model=original_load(self.path,**self.kwargs).eval()\n                for key,value in self.overrides.items():setattr(model,key,value)\n                object.__setattr__(self,\'loaded\',model)\n            return self.loaded\n        def eval(self):object.__setattr__(self,\'training\',False);return self\n        def to(self,*args,**kwargs):self._load().to(*args,**kwargs);return self\n        def cpu(self):\n            object.__setattr__(self,\'loaded\',None);gc.collect();torch.cuda.empty_cache();return self\n        def __call__(self,*args,**kwargs):return self._load()(*args,**kwargs)\n        def __getattr__(self,name):return getattr(self._load(),name)\n        def __setattr__(self,name,value):\n            if name in [\'path\',\'kwargs\',\'loaded\',\'overrides\']:object.__setattr__(self,name,value)\n            else:\n                self.overrides[name]=value\n                if self.loaded is not None:setattr(self.loaded,name,value)\n    def lazy_load(path,**kwargs):\n        marker=path.find(\'microsoft/\')\n        if marker<0:raise ValueError(\'Checkpoint outside pinned Microsoft models\')\n        return LazyModel(path[marker:],kwargs)\n    models.from_pretrained=lazy_load\n    from trellis2.modules.sparse.conv import config as conv_config\n    conv_config.FLEX_GEMM_ALGO=\'explicit_gemm\'\n    from flex_gemm.ops.spconv.submanifold_conv3d import SubMConv3dFunction\n    from flex_gemm.ops.spconv import Algorithm\n    import flex_gemm.ops.spconv as spconv\n    original_conv=SubMConv3dFunction._sparse_submanifold_conv_forward\n    def bounded_conv(feats,neighbor_cache,weight,bias=None):\n        if spconv.ALGORITHM!=Algorithm.EXPLICIT_GEMM:return original_conv(feats,neighbor_cache,weight,bias)\n        n=len(feats);co,kw,kh,kd,ci=weight.shape;volume=kw*kh*kd\n        neighbor=neighbor_cache[\'neighbor_map\'];flat_weight=weight.reshape(co,volume*ci).transpose(0,1)\n        output=torch.empty((n,co),device=feats.device,dtype=feats.dtype)\n        rows=max(64,min(8192,(128*2**20)//max(1,volume*ci*feats.element_size())))\n        for start in range(0,n,rows):\n            end=min(n,start+rows);ids=neighbor[start:end].reshape(-1);mask=ids!=0xffffffff\n            cols=torch.zeros(((end-start)*volume,ci),device=feats.device,dtype=feats.dtype)\n            cols[mask]=feats[ids.long()[mask]];cols=cols.reshape(end-start,volume*ci)\n            output[start:end]=torch.addmm(bias,cols,flat_weight) if bias is not None else cols@flat_weight\n        return output\n    SubMConv3dFunction._sparse_submanifold_conv_forward=staticmethod(bounded_conv)\n    report[\'adaptations\']=[\'BF16 attention evaluated in FP32 efficient kernel then returned as BF16\',\'per-sequence sparse SDPA\',\'stage-wise checkpoint reload from disk\',\'bounded explicit im2col\',\'preprocessed input: no background model\']\n    return original_sdpa\n')


In [ ]:
SOURCE_SHA='75fbf0183001ed9876c8dbb35de6b68552ee08bd'
MODEL_PINS={'microsoft/TRELLIS.2-4B': {'revision': 'af44b45f2e35a493886929c6d786e563ec68364d', 'files': [{'rfilename': '.gitattributes', 'size': 1519, 'lfs': None}, {'rfilename': 'README.md', 'size': 6417, 'lfs': None}, {'rfilename': 'ckpts/shape_dec_next_dc_f16c32_fp16.json', 'size': 678, 'lfs': None}, {'rfilename': 'ckpts/shape_dec_next_dc_f16c32_fp16.safetensors', 'size': 948490494, 'lfs': {'sha256': 'e3b718d3e43e4f8780e9a24ac6fff231811a67e3b058e336e10fe654c911d581', 'size': 948490494, 'pointerSize': 134}}, {'rfilename': 'ckpts/shape_enc_next_dc_f16c32_fp16.json', 'size': 653, 'lfs': None}, {'rfilename': 'ckpts/shape_enc_next_dc_f16c32_fp16.safetensors', 'size': 708797208, 'lfs': {'sha256': 'f37c5ff5b983b68e9946060000f09bc131f3e84318a2c8b7430a81e4b4636c41', 'size': 708797208, 'pointerSize': 134}}, {'rfilename': 'ckpts/slat_flow_img2shape_dit_1_3B_1024_bf16.json', 'size': 458, 'lfs': None}, {'rfilename': 'ckpts/slat_flow_img2shape_dit_1_3B_1024_bf16.safetensors', 'size': 2584574424, 'lfs': {'sha256': '07cd0596f634c5adc1890023d16023afc5eed02fb84b22bb23aff5bf0030fbbd', 'size': 2584574424, 'pointerSize': 135}}, {'rfilename': 'ckpts/slat_flow_img2shape_dit_1_3B_512_bf16.json', 'size': 458, 'lfs': None}, {'rfilename': 'ckpts/slat_flow_img2shape_dit_1_3B_512_bf16.safetensors', 'size': 2584574424, 'lfs': {'sha256': 'ec5e0917ef9b7e25ad51dffc7d19687a42019871f94239f2fa7f86264c55b70f', 'size': 2584574424, 'pointerSize': 135}}, {'rfilename': 'ckpts/slat_flow_imgshape2tex_dit_1_3B_1024_bf16.json', 'size': 458, 'lfs': None}, {'rfilename': 'ckpts/slat_flow_imgshape2tex_dit_1_3B_1024_bf16.safetensors', 'size': 2584672728, 'lfs': {'sha256': '580401269059a339b8318ab9ced459a13ba63391721c83a6c383198c29e77686', 'size': 2584672728, 'pointerSize': 135}}, {'rfilename': 'ckpts/slat_flow_imgshape2tex_dit_1_3B_512_bf16.json', 'size': 458, 'lfs': None}, {'rfilename': 'ckpts/slat_flow_imgshape2tex_dit_1_3B_512_bf16.safetensors', 'size': 2584672728, 'lfs': {'sha256': '8371aa1c5d13be79dcd5ddfd2cf3835e902e204dc34427169a1c702828e1a94d', 'size': 2584672728, 'pointerSize': 135}}, {'rfilename': 'ckpts/ss_flow_img_dit_1_3B_64_bf16.json', 'size': 467, 'lfs': None}, {'rfilename': 'ckpts/ss_flow_img_dit_1_3B_64_bf16.safetensors', 'size': 2584426920, 'lfs': {'sha256': 'ca01377c485bec418076d38ee80166d32dc776d744f2553b835cba1e97a7abf6', 'size': 2584426920, 'pointerSize': 135}}, {'rfilename': 'ckpts/tex_dec_next_dc_f16c32_fp16.json', 'size': 705, 'lfs': None}, {'rfilename': 'ckpts/tex_dec_next_dc_f16c32_fp16.safetensors', 'size': 948458812, 'lfs': {'sha256': '97ea69addea2ecd9312910f5f548234665eef51c088386180b7cd5b258645e3c', 'size': 948458812, 'pointerSize': 134}}, {'rfilename': 'ckpts/tex_enc_next_dc_f16c32_fp16.json', 'size': 676, 'lfs': None}, {'rfilename': 'ckpts/tex_enc_next_dc_f16c32_fp16.safetensors', 'size': 708797208, 'lfs': {'sha256': 'dd109f75f84b90fa411554ed6b0e4a87f430841163156fc0ebda2ebdc4752493', 'size': 708797208, 'pointerSize': 134}}, {'rfilename': 'pipeline.json', 'size': 4186, 'lfs': None}, {'rfilename': 'texturing_pipeline.json', 'size': 2965, 'lfs': None}]}, 'microsoft/TRELLIS-image-large': {'revision': '25e0d31ffbebe4b5a97464dd851910efc3002d96', 'files': [{'rfilename': '.gitattributes', 'size': 1519, 'lfs': None}, {'rfilename': 'README.md', 'size': 464, 'lfs': None}, {'rfilename': 'ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.json', 'size': 843, 'lfs': None}, {'rfilename': 'ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.safetensors', 'size': 171450952, 'lfs': {'sha256': '38c84bcef5ce0af1f48b1b5558dabc7575a13346043c41a7e0610f1fa619a161', 'size': 171450952, 'pointerSize': 134}}, {'rfilename': 'ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.json', 'size': 372, 'lfs': None}, {'rfilename': 'ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.safetensors', 'size': 181903412, 'lfs': {'sha256': '3e87aba94b5786407eb06d0502c1ed0885a0027a3f2b8537bfe15b0a92c01859', 'size': 181903412, 'pointerSize': 134}}, {'rfilename': 'ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.json', 'size': 396, 'lfs': None}, {'rfilename': 'ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.safetensors', 'size': 171450488, 'lfs': {'sha256': '737da6578d01948016b7c39786113af0d64a46f7922f6b8b5e698b84643be514', 'size': 171450488, 'pointerSize': 134}}, {'rfilename': 'ckpts/slat_enc_swin8_B_64l8_fp16.json', 'size': 321, 'lfs': None}, {'rfilename': 'ckpts/slat_enc_swin8_B_64l8_fp16.safetensors', 'size': 173242816, 'lfs': {'sha256': '21dceac6bee917ab6458ff52c9757ba89a779d03031c7bd17f9e7f0103bfd436', 'size': 173242816, 'pointerSize': 134}}, {'rfilename': 'ckpts/slat_flow_img_dit_L_64l8p2_fp16.json', 'size': 442, 'lfs': None}, {'rfilename': 'ckpts/slat_flow_img_dit_L_64l8p2_fp16.safetensors', 'size': 1203755136, 'lfs': {'sha256': '693fb2a58ad497bd222007301eeec49d14d60f8c12d2f2f00c221fa747b4c66c', 'size': 1203755136, 'pointerSize': 135}}, {'rfilename': 'ckpts/ss_dec_conv3d_16l8_fp16.json', 'size': 245, 'lfs': None}, {'rfilename': 'ckpts/ss_dec_conv3d_16l8_fp16.safetensors', 'size': 147591972, 'lfs': {'sha256': '1c76d4a40519aa2d711cc263a8404105231ac26db31d946bed48b84fee79009a', 'size': 147591972, 'pointerSize': 134}}, {'rfilename': 'ckpts/ss_enc_conv3d_16l8_fp16.json', 'size': 244, 'lfs': None}, {'rfilename': 'ckpts/ss_enc_conv3d_16l8_fp16.safetensors', 'size': 119068016, 'lfs': {'sha256': '107874eeaa0feb82f51b19db5da7db534fb7e7f19e5a122b9ff1bc2e258bfc6d', 'size': 119068016, 'pointerSize': 134}}, {'rfilename': 'ckpts/ss_flow_img_dit_L_16l8_fp16.json', 'size': 385, 'lfs': None}, {'rfilename': 'ckpts/ss_flow_img_dit_L_16l8_fp16.safetensors', 'size': 1130770840, 'lfs': {'sha256': '96dc6bfd4136fd950af564dd16b4ae533c9ba6af8f26c670646b2a9f2789b1db', 'size': 1130770840, 'pointerSize': 135}}, {'rfilename': 'pipeline.json', 'size': 1987, 'lfs': None}]}}
EXPECTED_WHEELS={'cumesh-0.0.1-cp313-cp313-linux_x86_64.whl': 'a60c661c67d19b96648c98c8d7a6bb80bcd190b446fac4afe01e9b5b0b01650e', 'nvdiffrast-0.4.0-cp313-cp313-linux_x86_64.whl': '992d930c68af3c40f9261d81adbd10dc53b1ee7dd5e97b67b3762f309c4be3c0', 'o_voxel-0.0.1-cp313-cp313-linux_x86_64.whl': '10f987ebce9c78c47a269b6321a00a1ad05faa312b31b8151a19b92c7b9fa38c', 'flex_gemm-1.0.0-cp313-cp313-linux_x86_64.whl': '07b57660b62a728c8e44a7902127127a8e28f2b548cfd274fd3f3f6a3573527c'}
PIPELINE_TYPE='512'
VALIDATION_ONLY=False
USE_CACHED_INPUT=True
RESUME_SHAPE=True
SHAPE_CHECKPOINT_SHA='32944f6bdaaa9e78ea9d17858d08b696e149263dc2b2d8b19af242310a0c214b'
ADAPTATION_FILE='/kaggle/temp/t4-adaptation.py'
"""Private bounded TRELLIS.2 generation, with original assets saved before export."""
import datetime,gc,hashlib,json,os,pathlib,subprocess,sys,time,traceback
START=time.monotonic();OUT=pathlib.Path('/kaggle/working/dragon-generation');OUT.mkdir(parents=True,exist_ok=True)
SCRATCH=pathlib.Path('/kaggle/temp/dragon-generation');SCRATCH.mkdir(parents=True,exist_ok=True)
for name in ['tmp','hf','pip']:(SCRATCH/name).mkdir(exist_ok=True)
os.environ.update(TMPDIR=str(SCRATCH/'tmp'),HF_HOME=str(SCRATCH/'hf'),PIP_CACHE_DIR=str(SCRATCH/'pip'),TORCH_CUDA_ARCH_LIST='7.5',ATTN_BACKEND='sdpa',SPARSE_ATTN_BACKEND='sdpa',SPARSE_CONV_BACKEND='flex_gemm',PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',OPENCV_IO_ENABLE_OPENEXR='1')
REPORT={'started_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),'stage':'setup','art_accepted':False,'pipeline_type':PIPELINE_TYPE,'seed':38017,'source_sha':SOURCE_SHA,'model_revisions':{k:v['revision'] for k,v in MODEL_PINS.items()},'phase_seconds':{}}
def save():
    REPORT['elapsed_seconds']=round(time.monotonic()-START,2)
    (OUT/'generation.json').write_text(json.dumps(REPORT,indent=2),encoding='utf-8');print(json.dumps(REPORT),flush=True)
def command(args,label,timeout,cwd=None):
    with (OUT/(label+'.log')).open('w') as log:result=subprocess.run(args,cwd=cwd,stdout=log,stderr=subprocess.STDOUT,timeout=timeout)
    if result.returncode:raise RuntimeError(label+': '+(OUT/(label+'.log')).read_text()[-3000:])
try:
    import torch
    if not torch.cuda.is_available():raise RuntimeError('GPU missing')
    REPORT.update(torch=torch.__version__,gpu=torch.cuda.get_device_name(),vram_gib=torch.cuda.get_device_properties(0).total_memory/2**30);save()
    command([sys.executable,'-m','pip','install','trimesh','opencv-python-headless','pillow','scipy','zstandard','tqdm','plyfile','easydict','transformers==4.56.2','safetensors','imageio','imageio-ffmpeg','kornia','timm'],'python-dependencies',240)
    command([sys.executable,'-m','pip','install','git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],'utils3d',180)
    checked=[]
    for name,sha in EXPECTED_WHEELS.items():
        wheel=next(pathlib.Path('/kaggle/input').rglob(name))
        if hashlib.sha256(wheel.read_bytes()).hexdigest()!=sha:raise RuntimeError('Wheel checksum mismatch')
        checked.append(str(wheel))
    command([sys.executable,'-m','pip','install','--no-deps',*checked],'preserved-wheels',90)
    repo=SCRATCH/'TRELLIS.2'
    command(['git','clone','--filter=blob:none','https://github.com/microsoft/TRELLIS.2.git',str(repo)],'clone',120)
    command(['git','checkout',SOURCE_SHA],'checkout',60,cwd=repo)
    import importlib.util
    spec=importlib.util.spec_from_file_location('t4_adapter',ADAPTATION_FILE);adapter=importlib.util.module_from_spec(spec);spec.loader.exec_module(adapter)
    adapter.patch_source(repo);sys.path.insert(0,str(repo))
    original_sdpa=adapter.install_runtime(MODEL_PINS,REPORT,save)
    REPORT['stage']='sparse_attention_validation';save()
    from trellis2.modules.sparse import VarLenTensor
    from trellis2.modules.sparse.attention.full_attn import sparse_scaled_dot_product_attention
    seq=[torch.randn((n,3,4,64),device='cuda',dtype=torch.bfloat16) for n in [17,29]]
    actual=sparse_scaled_dot_product_attention(VarLenTensor.from_tensor_list(seq)).to_tensor_list()
    errors=[]
    for qkv,got in zip(seq,actual):
        q,k,v=qkv.unbind(1)
        expected=original_sdpa(q.transpose(0,1).unsqueeze(0).float(),k.transpose(0,1).unsqueeze(0).float(),v.transpose(0,1).unsqueeze(0).float()).squeeze(0).transpose(0,1)
        if not torch.isfinite(got).all():raise RuntimeError('Nonfinite sparse attention')
        errors.append((got.float()-expected).abs().max().item())
    REPORT['attention_validation']={'max_abs_vs_fp32':errors,'dtype':'BF16'}
    if max(errors)>.04:raise RuntimeError('Attention reference mismatch')
    del seq,actual;torch.cuda.empty_cache()
    # Confirm the bounded im2col implementation against its original matrix expression.
    from flex_gemm.ops.spconv.submanifold_conv3d import SubMConv3dFunction
    import flex_gemm.ops.spconv as spconv
    spconv.set_algorithm('explicit_gemm')
    conv_errors=[]
    for dtype in [torch.float16,torch.bfloat16]:
        feats=torch.randn((9000,8),device='cuda',dtype=dtype)
        ids=torch.randint(0,len(feats),(9000,27),device='cuda',dtype=torch.int64)
        mask=torch.rand(ids.shape,device='cuda')>.15
        ids[~mask]=0xffffffff
        weight=torch.randn((12,3,3,3,8),device='cuda',dtype=dtype)*.1
        bias=torch.randn(12,device='cuda',dtype=dtype)
        got=SubMConv3dFunction._sparse_submanifold_conv_forward(feats,{'neighbor_map':ids},weight,bias)
        cols=torch.zeros((9000*27,8),device='cuda',dtype=dtype)
        cols[mask.reshape(-1)]=feats[ids.reshape(-1)[mask.reshape(-1)]]
        expected=torch.addmm(bias,cols.reshape(9000,27*8),weight.reshape(12,27*8).transpose(0,1))
        error=(got.float()-expected.float()).abs().max().item()
        if not torch.isfinite(got).all() or error>.04:raise RuntimeError('Bounded convolution reference mismatch')
        conv_errors.append({'dtype':str(dtype),'max_abs':error})
    REPORT['bounded_convolution_validation']=conv_errors
    del feats,ids,mask,weight,bias,got,cols,expected;torch.cuda.empty_cache();save()
    if VALIDATION_ONLY:
        from trellis2.pipelines import Trellis2ImageTo3DPipeline
        from transformers import DINOv3ViTModel
        REPORT['stage']='adaptation_preflight_completed';save();raise SystemExit(0)
    source=next(pathlib.Path('/kaggle/input').rglob('input-manifest.json')).parent
    manifest=json.loads((source/'input-manifest.json').read_text())
    dino=source/'dino'
    if not dino.is_dir():
        import zipfile
        archive=source/'dino.zip';dino=SCRATCH/'dino';dino.mkdir()
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                target=(dino/entry.filename).resolve()
                if not target.is_relative_to(dino.resolve()):raise RuntimeError('Unsafe input archive path')
            z.extractall(dino)
        if not (dino/'config.json').exists() and (dino/'dino'/'config.json').exists():dino=dino/'dino'
    if USE_CACHED_INPUT and not (dino/'model.safetensors').exists():
        import shutil
        writable=SCRATCH/'licensed-dino-config';shutil.copytree(dino,writable);dino=writable
        candidates=[p for p in pathlib.Path('/kaggle/input').rglob('model.safetensors') if p.stat().st_size==manifest['files']['model.safetensors']['bytes']]
        approved=None
        for candidate in candidates:
            if hashlib.sha256(candidate.read_bytes()).hexdigest()==manifest['files']['model.safetensors']['sha256']:approved=candidate;break
        if approved is None:raise RuntimeError('No cached DINO weight matches the authorized official SHA')
        (dino/'model.safetensors').symlink_to(approved)
        REPORT['dino_copy']={'server_source':'hark99/facebookdinov3-vitl16-pretrain-lvd1689m/transformers/default/1','original_official_sha_verified':True,'license':'original Meta DINOv3 license; authorized account'}
    if hashlib.sha256((dino/'model.safetensors').read_bytes()).hexdigest()!=manifest['files']['model.safetensors']['sha256']:raise RuntimeError('DINO input checksum mismatch')
    # Resolve config at the pinned public revision; keep credentials out of this notebook.
    import huggingface_hub
    config_file=huggingface_hub.hf_hub_download('microsoft/TRELLIS.2-4B','pipeline.json')
    config=json.loads(pathlib.Path(config_file).read_text());args=config['args']
    args['image_cond_model']['args']['model_name']=str(dino);args['low_vram']=True
    args['models']={k:(v if v.startswith('microsoft/') else 'microsoft/TRELLIS.2-4B/'+v) for k,v in args['models'].items()}
    if PIPELINE_TYPE=='512':args['models']={k:v for k,v in args['models'].items() if not k.endswith('_1024')}
    else:args['models'].pop('tex_slat_flow_model_512',None)
    config_dir=SCRATCH/'config';config_dir.mkdir();(config_dir/'pipeline.json').write_text(json.dumps(config))
    from trellis2.pipelines import Trellis2ImageTo3DPipeline
    from PIL import Image
    pipeline=Trellis2ImageTo3DPipeline.from_pretrained(str(config_dir));pipeline.cuda()
    for name in ['get_cond','sample_sparse_structure','sample_shape_slat','sample_shape_slat_cascade','decode_shape_slat','sample_tex_slat','decode_tex_slat']:
        original=getattr(pipeline,name)
        def hook(*a,_name=name,_original=original,**kw):
            REPORT['stage']=_name;save();start=time.monotonic();value=_original(*a,**kw)
            REPORT['phase_seconds'][_name]=REPORT['phase_seconds'].get(_name,0)+round(time.monotonic()-start,2)
            if _name in ['sample_shape_slat','sample_shape_slat_cascade','sample_tex_slat']:
                slat=value[0] if isinstance(value,tuple) else value
                checkpoint={'feats':slat.feats.cpu(),'coords':slat.coords.cpu()}
                checkpoint['cpu_rng_state']=torch.get_rng_state();checkpoint['cuda_rng_state']=torch.cuda.get_rng_state()
                if isinstance(value,tuple):checkpoint['resolution']=value[1]
                torch.save(checkpoint,OUT/(_name+'-checkpoint.pt'))
                REPORT.setdefault('checkpoints',[]).append(_name+'-checkpoint.pt')
            if _name=='decode_shape_slat':
                import trimesh
                raw=value[0][0]
                trimesh.Trimesh(raw.vertices.cpu().numpy(),raw.faces.cpu().numpy(),process=False).export(OUT/'decoded-shape-checkpoint.ply')
                REPORT.setdefault('checkpoints',[]).append('decoded-shape-checkpoint.ply')
            save();return value
        setattr(pipeline,name,hook)
    image=Image.open(source/'generation-input.png').convert('RGB')
    REPORT['input_sha256']=hashlib.sha256((source/'generation-input.png').read_bytes()).hexdigest()
    REPORT['stage']='generating';save()
    if RESUME_SHAPE:
        checkpoint=next(p for p in pathlib.Path('/kaggle/input').rglob('sample_shape_slat-checkpoint.pt') if hashlib.sha256(p.read_bytes()).hexdigest()==SHAPE_CHECKPOINT_SHA)
        saved=torch.load(checkpoint,map_location='cpu',weights_only=True)
        from trellis2.modules.sparse import SparseTensor
        shape=SparseTensor(feats=saved['feats'].cuda(),coords=saved['coords'].cuda())
        REPORT['resume']={'source_kernel':'gualwells/hostile-dragon-generation-20261011/1','shape_checkpoint_sha256':SHAPE_CHECKPOINT_SHA,'geometry_resampled':False}
        with torch.no_grad():
            condition=pipeline.get_cond([image],512)
            if 'cpu_rng_state' in saved:
                torch.set_rng_state(saved['cpu_rng_state']);torch.cuda.set_rng_state(saved['cuda_rng_state']);REPORT['resume']['rng_restored']=True
            else:torch.manual_seed(38018);REPORT['resume']['texture_seed']=38018
            tex=pipeline.sample_tex_slat(condition,pipeline.models['tex_slat_flow_model_512'],shape)
            meshes=pipeline.decode_latent(shape,tex,512)
        latent=(shape,tex,512)
    else:meshes,latent=pipeline.run(image,seed=38017,preprocess_image=False,pipeline_type=PIPELINE_TYPE,return_latent=True)
    shape,tex,res=latent
    torch.save({'shape_feats':shape.feats.cpu(),'shape_coords':shape.coords.cpu(),'tex_feats':tex.feats.cpu(),'tex_coords':tex.coords.cpu(),'resolution':res},OUT/'latents.pt')
    mesh=meshes[0]
    import numpy as np,trimesh
    vertices=mesh.vertices.cpu().numpy();faces=mesh.faces.cpu().numpy()
    trimesh.Trimesh(vertices,faces,process=False).export(OUT/'source-geometry.ply')
    np.savez_compressed(OUT/'source-volume.npz',vertices=vertices,faces=faces,attrs=mesh.attrs.cpu().numpy(),coords=mesh.coords.cpu().numpy(),voxel_size=mesh.voxel_size)
    (OUT/'source-volume-layout.json').write_text(json.dumps({'origin':mesh.origin.tolist(),'voxel_shape':list(mesh.voxel_shape),'layout':{k:[v.start,v.stop,v.step] for k,v in mesh.layout.items()}},indent=2))
    REPORT.update(actual_resolution=res,vertices=len(vertices),faces=len(faces),source_assets_saved=True);REPORT['stage']='exporting';save()
    import o_voxel.postprocess
    glb=o_voxel.postprocess.to_glb(vertices=mesh.vertices,faces=mesh.faces,attr_volume=mesh.attrs,coords=mesh.coords,attr_layout=mesh.layout,voxel_size=mesh.voxel_size,aabb=[[-.5,-.5,-.5],[.5,.5,.5]],decimation_target=500000,texture_size=2048 if PIPELINE_TYPE=='512' else 4096,remesh=False,verbose=True)
    target=OUT/'hostile-dragon-source.glb';glb.export(target,extension_webp=True)
    REPORT['glb']={'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()}
    REPORT['stage']='editable_candidate_saved'
except Exception:REPORT['stage']='generation_failed';REPORT['error']=traceback.format_exc()[-6000:]
finally:save()
